In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Labs"
)

xpt_file = DATA_DIR / "HDL_I.xpt"
csv_file = DATA_DIR / "HDL_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 HDL_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBDHDD",
    "LBDHDDSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HDL_I columns/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 8021:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,021 rows x 3 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# HDL_I is a MIXED integer / decimal laboratory file.
#
#
# SEQN
#     Respondent sequence number.
#
# LBDHDD
#     Direct HDL cholesterol in mg/dL.
#     Released as whole-number values.
#
# LBDHDDSI
#     Direct HDL cholesterol in mmol/L.
#     Genuine continuous decimal field.
#
#
# Therefore:
#
#     SEQN      -> Int64
#     LBDHDD    -> Int64
#     LBDHDDSI  -> KEEP FLOAT
#
#
# IMPORTANT:
#
# Do NOT globally remove ".0" from all three fields.
#
# LBDHDDSI is a genuine decimal variable and values
# such as 3.0 are legitimate.
#

DECIMAL_COLUMNS = [
    "LBDHDDSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "LBDHDD"
]


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly 1 genuine decimal field."
    )


if len(INTEGER_COLUMNS) != 2:

    raise ValueError(
        "Expected exactly 2 integer/ID fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/ID fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero representation:
#
#     5.397605346934028e-79
#
# Attached HDL_I contains NONE.
#
# Therefore:
#
#     DO NOT replace anything with zero.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all HDL_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total > 0:

    print(
        "\nUnexpected tiny values:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HDL_I."
    )


if true_zero_total > 0:

    print(
        "\nUnexpected zeros:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary zero values "
        "found in HDL_I."
    )


print(
    "PASS: HDL_I contains 0 tiny artifacts "
    "and 0 ordinary zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# No XPORT zero corruption was found.
#
# Therefore every numeric source value should remain
# exactly as released.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero correction required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE converting anything to Int64.
#
# np.round() below is ONLY a diagnostic comparison.
#
# It does NOT alter source values.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: LBDHDDSI is the only genuine "
    "decimal field."
)


# ============================================================
# 8. LBDHDDSI DECIMAL DETAILS
# ============================================================

s = df["LBDHDDSI"]


si_nonmissing = int(
    s.notna()
    .sum()
)


si_missing = int(
    s.isna()
    .sum()
)


si_values = (
    s.dropna()
    .to_numpy(
        dtype=float
    )
)


si_fractional = int(
    (
        ~np.isclose(
            si_values,
            np.round(si_values),
            rtol=0,
            atol=1e-12
        )
    )
    .sum()
)


si_whole_looking = (
    si_nonmissing
    -
    si_fractional
)


print("\n--- LBDHDDSI DECIMAL DETAILS ---")

print(
    "Nonmissing:",
    f"{si_nonmissing:,}"
)

print(
    "Missing:",
    f"{si_missing:,}"
)

print(
    "Minimum:",
    repr(float(s.min()))
)

print(
    "Maximum:",
    repr(float(s.max()))
)

print(
    "Fractional observations:",
    f"{si_fractional:,}"
)

print(
    "Whole-looking observations:",
    f"{si_whole_looking:,}"
)


if si_nonmissing != 7256:

    raise ValueError(
        "Unexpected LBDHDDSI nonmissing count."
    )


if si_missing != 765:

    raise ValueError(
        "Unexpected LBDHDDSI missing count."
    )


if float(s.min()) != 0.16:

    raise ValueError(
        "Unexpected LBDHDDSI minimum."
    )


if float(s.max()) != 5.84:

    raise ValueError(
        "Unexpected LBDHDDSI maximum."
    )


if si_fractional != 7255:

    raise ValueError(
        "Unexpected number of fractional "
        "LBDHDDSI observations."
    )


if si_whole_looking != 1:

    raise ValueError(
        "Unexpected number of whole-looking "
        "LBDHDDSI values."
    )


print(
    "PASS: LBDHDDSI decimal structure validated."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 10. VALIDATE LBDHDD IS WHOLE-NUMBER
# ============================================================

#
# LBDHDD is HDL cholesterol in mg/dL.
#
# In this released file all populated results are
# whole-number values.
#
# We prove this before Int64 conversion.
#

hdd_values = (
    df["LBDHDD"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


hdd_whole_mask = np.isclose(
    hdd_values,
    np.round(hdd_values),
    rtol=0,
    atol=1e-12
)


if not hdd_whole_mask.all():

    raise ValueError(
        "LBDHDD unexpectedly contains "
        "fractional values."
    )


print(
    "PASS: Every populated LBDHDD value "
    "is whole-number."
)


# ============================================================
# 11. CONVERT LBDHDD TO NULLABLE Int64
# ============================================================

#
# NO round() is used during conversion.
#

df["LBDHDD"] = (
    df["LBDHDD"]
    .astype("Int64")
)


print(
    "PASS: LBDHDD converted to nullable Int64 "
    "without rounding."
)


# ============================================================
# 12. CONFIRM LBDHDDSI REMAINS FLOAT
# ============================================================

if not pd.api.types.is_float_dtype(
    df["LBDHDDSI"].dtype
):

    raise ValueError(
        "LBDHDDSI is no longer floating-point."
    )


print(
    "PASS: LBDHDDSI remains floating-point."
)


# ============================================================
# 13. CDC CHECK - LBDHDD
# ============================================================

#
# CDC:
#
# Range:
#     6 to 226 mg/dL
#
# Nonmissing:
#     7,256
#
# Missing:
#     765
#

hdd_nonmissing = int(
    df["LBDHDD"]
    .notna()
    .sum()
)


hdd_missing = int(
    df["LBDHDD"]
    .isna()
    .sum()
)


hdd_range_count = int(
    df["LBDHDD"]
    .between(
        6,
        226
    )
    .sum()
)


hdd_min = int(
    df["LBDHDD"]
    .min()
)


hdd_max = int(
    df["LBDHDD"]
    .max()
)


print("\n--- LBDHDD ---")

print(
    "Nonmissing:",
    f"{hdd_nonmissing:,}"
)

print(
    "Missing:",
    f"{hdd_missing:,}"
)

print(
    "Range count:",
    f"{hdd_range_count:,}"
)

print(
    "Minimum:",
    hdd_min
)

print(
    "Maximum:",
    hdd_max
)


if hdd_nonmissing != 7256:

    raise ValueError(
        "LBDHDD nonmissing count differs."
    )


if hdd_missing != 765:

    raise ValueError(
        "LBDHDD missing count differs."
    )


if hdd_range_count != 7256:

    raise ValueError(
        "LBDHDD range count differs."
    )


if hdd_min != 6:

    raise ValueError(
        "LBDHDD minimum differs."
    )


if hdd_max != 226:

    raise ValueError(
        "LBDHDD maximum differs."
    )


print(
    "PASS: LBDHDD matches CDC."
)


# ============================================================
# 14. CDC CHECK - LBDHDDSI
# ============================================================

#
# CDC:
#
# Range:
#     0.16 to 5.84 mmol/L
#
# Nonmissing:
#     7,256
#
# Missing:
#     765
#

si_range_count = int(
    df["LBDHDDSI"]
    .between(
        0.16,
        5.84
    )
    .sum()
)


if si_range_count != 7256:

    raise ValueError(
        "LBDHDDSI range count differs."
    )


if int(
    df["LBDHDDSI"]
    .notna()
    .sum()
) != 7256:

    raise ValueError(
        "LBDHDDSI nonmissing count differs."
    )


if int(
    df["LBDHDDSI"]
    .isna()
    .sum()
) != 765:

    raise ValueError(
        "LBDHDDSI missing count differs."
    )


print(
    "PASS: LBDHDDSI matches CDC range/count."
)


# ============================================================
# 15. LBDHDD / LBDHDDSI MISSING-MASK CHECK
# ============================================================

#
# Both values represent the same HDL result in
# different units.
#
# Missing positions should match exactly.
#

hdd_missing_mask = (
    df["LBDHDD"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


si_missing_mask = (
    df["LBDHDDSI"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


mask_mismatches = int(
    np.sum(
        hdd_missing_mask
        !=
        si_missing_mask
    )
)


print(
    "\n--- HDL MISSING MASK CHECK ---"
)

print(
    "Mask mismatches:",
    mask_mismatches
)


if mask_mismatches != 0:

    raise ValueError(
        "LBDHDD and LBDHDDSI missing "
        "positions differ."
    )


print(
    "PASS: LBDHDD and LBDHDDSI missing "
    "positions match exactly."
)


# ============================================================
# 16. CDC SI CONVERSION CHECK
# ============================================================

#
# CDC documentation states:
#
#     HDL mg/dL * 0.02586 = mmol/L
#
# The released SI values are shown to two decimals.
#
#
# IMPORTANT:
#
# np.round() below is ONLY used on a TEMPORARY validation
# calculation.
#
# We DO NOT overwrite LBDHDDSI.
#
# The released XPT value remains the fidelity source.
#

valid_mask = (
    df["LBDHDD"]
    .notna()
    &
    df["LBDHDDSI"]
    .notna()
)


hdd_for_formula = (
    df.loc[
        valid_mask,
        "LBDHDD"
    ]
    .to_numpy(
        dtype=float
    )
)


released_si = (
    df.loc[
        valid_mask,
        "LBDHDDSI"
    ]
    .to_numpy(
        dtype=float
    )
)


calculated_si = np.round(
    hdd_for_formula * 0.02586,
    2
)


si_formula_mismatches = int(
    np.sum(
        calculated_si
        !=
        released_si
    )
)


print(
    "\n--- SI CONVERSION CHECK ---"
)

print(
    "Formula-tested records:",
    f"{len(released_si):,}"
)

print(
    "Conversion mismatches:",
    si_formula_mismatches
)


if si_formula_mismatches != 0:

    raise ValueError(
        "Released LBDHDDSI values do not match "
        "the documented HDL conversion."
    )


print(
    "PASS: All 7,256 SI values match "
    "mg/dL x 0.02586 at released precision."
)


# ============================================================
# 17. CONFIRM NO ZEROS CREATED
# ============================================================

#
# Original XPT contained:
#
#     tiny values = 0
#     ordinary zeros = 0
#
# Therefore cleaning must not create zeros.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 18. NUMERIC FIDELITY BEFORE EXPORT
# ============================================================

#
# HDL_I required NO numeric correction.
#
# Therefore all three variables must remain numerically
# identical to the original XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 3 variables preserve "
    "the source XPT values exactly."
)


# ============================================================
# 19. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY EXPORT:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# Therefore:
#
# SEQN
#     83732.0 -> 83732
#
# LBDHDD
#     46.0 -> 46
#
# LBDHDDSI
#     1.19 -> 1.19
#     3.0  -> 3.0
#
# The genuine decimal field stays floating point.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 20. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 21. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: 0 numeric differences across "
    "all 3 HDL_I variables."
)


# ============================================================
# 22. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 23. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 24. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 25. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT include LBDHDDSI.
#
# It is a genuine continuous decimal lab field.
#
# Only:
#
#     SEQN
#     LBDHDD
#
# should be checked for unwanted ".0".
#

print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in integer fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "SEQN or LBDHDD."
)


# ============================================================
# 26. LEGITIMATE LBDHDDSI .0 VALUES
# ============================================================

#
# INFORMATIONAL ONLY.
#
# LBDHDDSI is a genuine continuous variable.
#
# The attached file contains one whole-looking
# SI value:
#
#     3.0
#
# It should stay:
#
#     3.0
#
# not be converted to integer 3.
#

si_dot_zero_count = int(
    (
        raw_csv_df["LBDHDDSI"]
        .ne("")
        &
        raw_csv_df["LBDHDDSI"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\n--- LEGITIMATE LBDHDDSI .0 VALUES ---"
)

print(
    "Count:",
    si_dot_zero_count
)


if si_dot_zero_count != 1:

    raise ValueError(
        "Unexpected LBDHDDSI '.0' token count."
    )


print(
    "PASS: The legitimate LBDHDDSI "
    "whole-looking decimal value remains 3.0."
)


# ============================================================
# 27. EXACT LBDHDDSI DECIMAL FIDELITY
# ============================================================

#
# rtol = 0
# atol = 0
#
# No numerical change is accepted.
#

expected_si = (
    expected_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


exported_si = (
    roundtrip_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


si_exact = np.allclose(
    expected_si,
    exported_si,
    rtol=0,
    atol=0,
    equal_nan=True
)


if not si_exact:

    raise ValueError(
        "LBDHDDSI decimal fidelity failed."
    )


print(
    "\nPASS: Every LBDHDDSI value "
    "preserved exactly."
)


# ============================================================
# 28. FINAL CSV RANGE CHECK
# ============================================================

final_hdd = pd.to_numeric(
    raw_csv_df["LBDHDD"]
    .where(
        raw_csv_df["LBDHDD"] != "",
        np.nan
    ),
    errors="coerce"
)


final_si = pd.to_numeric(
    raw_csv_df["LBDHDDSI"]
    .where(
        raw_csv_df["LBDHDDSI"] != "",
        np.nan
    ),
    errors="coerce"
)


if final_hdd.min() != 6:

    raise ValueError(
        "Final LBDHDD minimum differs."
    )


if final_hdd.max() != 226:

    raise ValueError(
        "Final LBDHDD maximum differs."
    )


if final_si.min() != 0.16:

    raise ValueError(
        "Final LBDHDDSI minimum differs."
    )


if final_si.max() != 5.84:

    raise ValueError(
        "Final LBDHDDSI maximum differs."
    )


print(
    "PASS: Final HDL ranges preserved."
)


# ============================================================
# 29. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "HDL_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Laboratory - "
    "High-Density Lipoprotein Cholesterol"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Whole-number/ID fields:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "LBDHDD nonmissing:",
    f"{hdd_nonmissing:,}"
)


print(
    "LBDHDD missing:",
    f"{hdd_missing:,}"
)


print(
    "LBDHDD range:",
    f"{hdd_min}-{hdd_max} mg/dL"
)


print(
    "LBDHDDSI nonmissing:",
    f"{si_nonmissing:,}"
)


print(
    "LBDHDDSI missing:",
    f"{si_missing:,}"
)


print(
    "LBDHDDSI range:",
    "0.16-5.84 mmol/L"
)


print(
    "SI conversion mismatches:",
    si_formula_mismatches
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "LBDHDDSI decimal differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields:",
    "0 expected"
)


print(
    "LBDHDDSI '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "LBDHDDSI remains a genuine decimal "
    "laboratory variable."
)


print(
    "No laboratory decimal value was rounded "
    "or rewritten."
)

NHANES 2015-2016 HDL_I XPT -> CSV
Rows: 8,021
Columns: 3
PASS: Structure = 8,021 rows x 3 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['LBDHDDSI']
Whole-number/ID fields: ['SEQN', 'LBDHDD']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: HDL_I contains 0 tiny artifacts and 0 ordinary zeros.

PASS: No tiny-to-zero correction required.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
LBDHDD: 0 fractional observations
LBDHDDSI: 7,255 fractional observations

PASS: LBDHDDSI is the only genuine decimal field.

--- LBDHDDSI DECIMAL DETAILS ---
Nonmissing: 7,256
Missing: 765
Minimum: 0.16
Maximum: 5.84
Fractional observations: 7,255
Whole-looking observations: 1
PASS: LBDHDDSI decimal structure validated.

--- SEQN ---
Minimum: 83732
Maximum: 93702
Missing: 0
Duplicates: 0
PASS: SEQN 83732-93702 validated.
PASS: Every populated LBDHDD value is whole-number.
PASS: LBDHDD converted to nullable Int64 without rounding.
PASS: